# 🚁 Entraînement drone YOLO11n sur GPU cloud (Google Colab)

**~1-2 h pour 100 epochs sur T4 gratuit**, au lieu de ~6-18 h sur un Mac M3.

Le script `train_drone.py` est inchangé : il détecte CUDA automatiquement, télécharge le dataset Roboflow, remappe les classes et entraîne.

## ⚠️ AVANT TOUT — activer le GPU
Menu **Exécution → Modifier le type d'exécution → Accélérateur matériel : GPU (T4)** → Enregistrer.

Puis exécute les cellules dans l'ordre (Maj+Entrée).

### 1. Vérifier que le GPU est bien actif
Si ça affiche `Tesla T4` (ou autre GPU), c'est bon. Sinon, retourne activer le GPU ci-dessus.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 2. Google Drive — **seulement** pour sauver le modèle final
⚠️ On **n'entraîne PAS sur Drive** : lire/écrire les ~30 000 fichiers du dataset via le montage Drive est extrêmement lent (chaque fichier = un aller-retour réseau → le remap et chaque epoch rament).

On travaille donc sur le **disque local de Colab (`/content`, rapide)**, et on copie juste le poids final (~5 Mo) dans Drive à la fin.

Contrepartie : si Colab déconnecte en cours de route, on relance (dataset re-téléchargé + entraînement ~1-2 h). Garde l'onglet actif.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/drone_training'   # uniquement le poids final
os.makedirs(DRIVE_OUT, exist_ok=True)

WORKDIR = '/content/drone_training'                   # disque LOCAL Colab = rapide
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print('Travail (local, rapide) :', os.getcwd())
print('Sauvegarde finale ->', DRIVE_OUT)

### 3. Installer les dépendances
PyTorch est **déjà installé sur Colab avec CUDA** (maintenu par Google, aligné sur les drivers GPU). On n'y touche surtout pas — on ajoute seulement `ultralytics` et `roboflow`. La cellule **3b** vérifie ensuite que rien n'a cassé CUDA.

In [ ]:
# torch_avant = version CUDA pré-installée par Colab ; on s'assure que pip ne la remplace pas.
import torch
torch_avant = torch.__version__
!pip install -q ultralytics roboflow pyyaml

### 3b. Vérifier que CUDA est bien actif ✅
Doit afficher `CUDA dispo : True` et le nom du GPU. Si c'est `False`, la cellule s'arrête et t'explique comment corriger (pip a remplacé torch, ou GPU pas activé).

In [ ]:
import importlib, torch
importlib.reload(torch)  # recharge au cas où pip aurait touché torch
print('torch      :', torch.__version__, '(avant pip :', torch_avant + ')')
print('CUDA build :', torch.version.cuda)
print('CUDA dispo :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))
    import ultralytics; ultralytics.checks()
else:
    raise SystemExit(
        'CUDA indisponible ! Causes possibles :\n'
        ' 1) GPU pas active -> Execution > Modifier le type d execution > GPU, puis relance.\n'
        ' 2) pip a installe un torch CPU. Corrige avec :\n'
        '      !pip install -q --force-reinstall torch torchvision\n'
        '    puis Execution > Redemarrer la session et relance depuis la cellule 1.'
    )

### 4. Récupérer `train_drone.py`
Exécute la cellule, puis clique **Choisir les fichiers** et sélectionne le `train_drone.py` de ton Mac (`tracker/training/train_drone.py`).

_(Alternative si ton repo est public : remplace cette cellule par `!git clone https://github.com/SkaosDev/EuropeanDefenseTechHackathon.git && cp EuropeanDefenseTechHackathon/tracker/training/train_drone.py .`)_

In [ ]:
from google.colab import files

if not os.path.exists('train_drone.py'):
    print('Sélectionne train_drone.py depuis ton Mac :')
    files.upload()
assert os.path.exists('train_drone.py'), 'train_drone.py manquant — relance la cellule.'
print('OK : train_drone.py prêt.')

### 5. Clé API Roboflow
Compte gratuit → https://app.roboflow.com → **Settings → API → Private API Key**.
Saisie masquée (non stockée dans le notebook).

### 6. Lancer l'entraînement 🚀
Session Colab gratuite = **coupable à tout moment** (max ~4h20, non garanti). Deux protections :
- `--project` pointe les poids vers **Drive** → `best.pt` y est mis à jour **à chaque epoch**, donc même une coupure ne te fait rien perdre.
- `--epochs 30` : le transfer learning converge vite (déjà mAP50 ≈ 0.87 dès l'epoch 1), 30 suffit largement et tient dans la fenêtre (~4 h). Mets `--epochs 25` pour plus de marge, ou `40` si tu te sens chanceux.

Garde l'onglet actif. Le dataset n'est téléchargé qu'une fois par session.

In [ ]:
# --project vers Drive : best.pt sauvegardé à chaque epoch (survit aux coupures Colab).
!python train_drone.py --epochs 30 --project "$DRIVE_OUT/runs"

### 7. Récupérer le modèle
- **Entraînement allé au bout** → `drone_yolo11n.pt` est créé ; la cellule ci-dessous le copie dans Drive + le télécharge.
- **Colab a coupé avant la fin** → pas grave : grâce à `--project`, le meilleur modèle est **déjà dans Drive** ici :
  `MyDrive/drone_training/runs/drone_yolo11n/weights/best.pt`. Renomme-le `drone_yolo11n.pt` et utilise-le tel quel.

In [ ]:
import os, shutil
from google.colab import files

best_drive = f'{DRIVE_OUT}/runs/drone_yolo11n/weights/best.pt'
if os.path.exists('drone_yolo11n.pt'):           # entraînement terminé proprement
    shutil.copy2('drone_yolo11n.pt', DRIVE_OUT)
    print('Copié dans Drive :', DRIVE_OUT + '/drone_yolo11n.pt')
    files.download('drone_yolo11n.pt')
elif os.path.exists(best_drive):                 # coupé avant la fin : on prend le best.pt
    shutil.copy2(best_drive, f'{DRIVE_OUT}/drone_yolo11n.pt')
    print('Session coupée mais modèle sauvé :', DRIVE_OUT + '/drone_yolo11n.pt')
    files.download(best_drive)
else:
    print('Aucun poids trouvé — l’entraînement n’a pas encore produit de best.pt.')

In [ ]:
from google.colab import files

print('Dispo dans Drive :', os.path.abspath('drone_yolo11n.pt'))
files.download('drone_yolo11n.pt')

Ensuite, sur ton Mac / le Pi : copie `drone_yolo11n.pt` dans `tracker/models/` (voir le README du dossier training).